In [ ]:
!pip install -q langchain langchain-openai langchain-core
!pip install -q langchain langchain-openai langchain-core python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.1/122.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 18.8 MB/s eta 0:00:00


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.tools import tool
from langchain_core.messages import ToolMessage
from langchain_core.messages import HumanMessage, ToolMessage

In [ ]:
OPENROUTER_API_KEY = ""
MODEL_NAME = "cohere/north-mini-code:free"

In [ ]:
llm = ChatOpenAI(
    model=MODEL_NAME,
    openai_api_key=OPENROUTER_API_KEY,
    openai_api_base="https://openrouter.ai/api/v1",
    temperature=0.2
)

#Challenge 1


In [ ]:
persona_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a {persona}. Explain concepts at a {target_level} level."),
    ("human", "Explain the topic: {topic}")
])

parser = StrOutputParser()

chain = persona_prompt | llm | parser

response = chain.invoke({
    "persona": "Quantum Physicist",
    "target_level": "high school",
    "topic": "Quantum Superposition"
})

print(response)


**Quantum Superposition – What It Is and Why It Matters**

---

### 1. The Classic Picture vs. Quantum Reality  

In everyday life we’re used to objects being in one definite state.  
- A ball is either **up** or **down** on a table.  
- A light switch is either **on** or **off**.  

Quantum mechanics tells us that at the tiniest scales (atoms, electrons, photons) things behave very differently. A quantum system can exist in **multiple states at once**—a condition called **superposition**.

---

### 2. A Simple Analogy: A Coin in a Box  

Imagine a magical coin that can be in three places at the same time:

| State | What it means |
|-------|----------------|
| **Heads** | The coin is flat on the table. |
| **Tails** | The coin is flat on the table, but flipped. |
| **In‑the‑air** | The coin is mid‑flip, not yet settled. |

If you look at the coin **without disturbing it**, you could say it is simultaneously **heads, tails, and in‑the‑air**. Only when you peek (measure) does the coin “

#Challenge 2

In [ ]:
summarize_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert at summarizing articles."),
    ("human", "Summarize the following article into exactly 2 bullet points:\n\n{article_text}")
])

parser = StrOutputParser()

summarize_chain = summarize_prompt | llm | parser


translate_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a professional French translator."),
    ("human", "Translate the following summary into French:\n\n{summary}")
])


final_chain = (
    {"summary": summarize_chain} | translate_prompt | llm | StrOutputParser()
)


response = final_chain.invoke({
    "article_text": """
    Artificial intelligence is transforming healthcare.
    It helps doctors diagnose diseases faster.
    """
})

print(response)

L’intelligence artificielle est en train de transformer le secteur de la santé.  
Elle permet aux médecins de diagnostiquer les maladies plus rapidement.


#Challenge 3

In [ ]:
@tool
def currency_converter(amount: float, from_curr: str, to_curr: str) -> float:
    """Convert an amount between USD, EUR, and GBP."""

    exchange_rates = {
        ("USD", "EUR"): 0.92,
        ("USD", "GBP"): 0.79,
        ("EUR", "USD"): 1.09,
        ("GBP", "USD"): 1.26
    }

    if from_curr == to_curr:
        return amount

    rate = exchange_rates.get((from_curr.upper(), to_curr.upper()))

    if rate is None:
        raise ValueError("Unsupported currency conversion.")

    return amount * rate

@tool
def calculate_compound_interest(principal: float, rate: float, years: int) -> float:
    """
    Calculate the final investment value after annual compound interest.
    Returns the total accumulated amount, including the original principal.
    """
    return principal * ((1 + rate) ** years)



llm_with_fin_tools = llm.bind_tools([
    currency_converter,
    calculate_compound_interest
])



response = llm_with_fin_tools.invoke(
    "Convert 250 USD to EUR."
)
print(response.tool_calls)



response = llm_with_fin_tools.invoke(
    "Calculate the compound interest on $5000 at 8% for 10 years."
)
print(response.tool_calls)


[{'name': 'currency_converter', 'args': {'amount': 250, 'from_curr': 'USD', 'to_curr': 'EUR'}, 'id': 'currency_converter_mqyqffx2fs9s', 'type': 'tool_call'}]
[{'name': 'calculate_compound_interest', 'args': {'principal': 5000.0, 'rate': 0.08, 'years': 10}, 'id': 'calculate_compound_interest_gedyxgegpeh3', 'type': 'tool_call'}]


#Challenge 4

In [ ]:
fin_tools = {
    "currency_converter": currency_converter,
    "calculate_compound_interest": calculate_compound_interest
}

def run_financial_agent(user_query: str) -> str:

    messages = [
        HumanMessage(content=user_query)
    ]


    response = llm_with_fin_tools.invoke(messages)


    if not response.tool_calls:
        return response.content


    messages.append(response)


    for tool_call in response.tool_calls:
        tool_name = tool_call["name"]
        tool = fin_tools[tool_name]


        result = tool.invoke(tool_call["args"])


        messages.append(
            ToolMessage(
                content=str(result),
                tool_call_id=tool_call["id"]
            )
        )


    final_response = llm_with_fin_tools.invoke(messages)

    return final_response.content

In [ ]:
print(run_financial_agent("Convert 250 USD to EUR."))

250 USD converts to **230.00 EUR** at the current exchange rate.


In [ ]:
print(run_financial_agent(
    "Calculate the compound interest on $5000 at 8% for 10 years."
))

I calculated the compound interest for you, but there appears to be an issue with the function output format. Let me provide you with the correct calculation:

**Compound Interest Calculation:**
- Principal: $5,000
- Annual Interest Rate: 8%
- Time Period: 10 years

**Formula:** A = P(1 + r)^t
- A = $5,000 × (1 + 0.08)^10
- A = $5,000 × (1.08)^10
- A = $5,000 × 2.1589
- **Final Amount: $10,794.50**

**Compound Interest Earned:** $10,794.50 - $5,000 = **$5,794.50**

The function seems to have returned an incorrectly formatted result. The correct final amount after 10 years at 8% compound interest on $5,000 should be approximately $10,794.50, meaning you would earn about $5,794.50 in interest over the 10-year period.


In [ ]:
print(run_financial_agent("What is compound interest?"))

Compound interest is a powerful financial concept where interest is calculated not only on the initial principal amount but also on the accumulated interest from previous periods. This creates a "snowball" effect where your money grows exponentially over time.

Here's how it works:

**Basic Formula:**
A = P(1 + r/n)^(nt)

Where:
- A = the future value of the investment/loan
- P = the principal amount (initial investment)
- r = annual interest rate (decimal)
- n = number of times interest is compounded per year
- t = number of years

**Key Characteristics:**
1. **Interest on interest**: Unlike simple interest (which only applies to the principal), compound interest applies to both the principal and previously earned interest
2. **Frequency matters**: The more frequently interest is compounded (daily, monthly, quarterly, annually), the faster your money grows
3. **Time is your ally**: The longer the investment period, the more dramatic the compounding effect becomes

**Example:**
If you 